# Case Study 2: Credit Card Fraud Detection

In this assignment we apply **XGBoost** to detect fraudulent credit card transactions using the IEEE-CIS Fraud Detection dataset from Kaggle.

Because fraud is rare, the dataset is heavily **imbalanced**. To deal with this, we will:
- Use **SMOTE** to oversample the minority (fraud) class in the training data only
- Train an **XGBoost** classifier
- Tune the **decision threshold** instead of relying on the default 0.5
- Interpret the model using **feature importance**

We only use `train_transaction.csv` and `train_identity.csv`. The test files and sample submission are not used, since we do not have labels for them.

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

## 2. Load Dataset

In [ ]:
# Load dataset
train_transaction = pd.read_csv("/content/train_transaction.csv")
train_identity = pd.read_csv("/content/train_identity.csv")

In [ ]:
print("Transaction data shape:", train_transaction.shape)
print("Identity data shape:", train_identity.shape)

## 3. Merge the Two Files

We merge the transaction data and the identity data using `TransactionID`. We use a **left join** so that we keep every transaction, even if it does not have matching identity information.

In [ ]:
# Merge transaction and identity data on TransactionID
fraud_data = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

In [ ]:
fraud_data.shape

## 4. Inspect the Dataset

Before doing anything else, let's look at the data: the first few rows, the column names, missing values, and how imbalanced the target variable (`isFraud`) is.

In [ ]:
# First 5 rows
print(fraud_data.head())

In [ ]:
# Column names
print(fraud_data.columns.tolist())

In [ ]:
# Missing values (top 20 columns with the most missing values)
print("Missing values:")
print(fraud_data.isnull().sum().sort_values(ascending=False).head(20))

In [ ]:
# Class distribution of the target variable
print("isFraud value counts:")
print(fraud_data["isFraud"].value_counts())

fraud_percentage = fraud_data["isFraud"].mean() * 100
print("Percentage of fraudulent transactions: {:.2f}%".format(fraud_percentage))

As we can see, only a small percentage of transactions are fraudulent. This confirms that the dataset is heavily imbalanced, which is why we will use SMOTE later and why we should not rely on accuracy alone to judge the model.

## 5. Handle Missing Values

The IEEE-CIS dataset has a lot of missing values, especially in the identity columns. We handle this in a simple way:
- For **numerical columns**, we fill missing values with the median.
- For **categorical (object) columns**, we fill missing values with the string `"Unknown"`.

This is a simple approach that keeps the assignment easy to follow, while still allowing the model to use these columns.

In [ ]:
# Separate numerical and categorical columns
numerical_cols = fraud_data.select_dtypes(include=[np.number]).columns
categorical_cols = fraud_data.select_dtypes(include=["object"]).columns

print("Number of numerical columns:", len(numerical_cols))
print("Number of categorical columns:", len(categorical_cols))

In [ ]:
# Fill missing numerical values with the median
for col in numerical_cols:
    fraud_data[col] = fraud_data[col].fillna(fraud_data[col].median())

# Fill missing categorical values with "Unknown"
for col in categorical_cols:
    fraud_data[col] = fraud_data[col].fillna("Unknown")

In [ ]:
# Confirm there are no missing values left
print("Total missing values after filling:", fraud_data.isnull().sum().sum())

## 6. Encode Categorical Variables

XGBoost needs numerical input, so we convert the categorical (object) columns into numbers using `pd.get_dummies()`. This creates a new binary (0/1) column for each category.

In [ ]:
# Identify categorical columns again (in case dtypes changed)
categorical_cols = fraud_data.select_dtypes(include=["object"]).columns
print("Categorical columns to encode:", list(categorical_cols))

In [ ]:
# One-hot encode categorical columns
fraud_data = pd.get_dummies(
    fraud_data,
    columns=categorical_cols,
    drop_first=True
)

In [ ]:
# Confirm that no object (string) columns remain
print("Remaining object columns:", fraud_data.select_dtypes(include=["object"]).columns.tolist())
print("New shape after encoding:", fraud_data.shape)

## 7. Separate Features and Target

We separate the target variable (`isFraud`) from the features. We also remove `TransactionID` because it is just an identifier and has no predictive meaning.

In [ ]:
# Separate features and target
X = fraud_data.drop("isFraud", axis=1)
y = fraud_data["isFraud"]

# Remove TransactionID since it is only an identifier
X = X.drop("TransactionID", axis=1)

In [ ]:
print("Features shape:", X.shape)
print("Target shape:", y.shape)

## 8. Train-Test Split

Since the dataset is heavily imbalanced, we use `stratify=y` so that the train and test sets both keep roughly the same fraud percentage as the original data.

In [ ]:
# Split data (stratified because the target is imbalanced)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=1,
    stratify=y
)

In [ ]:
# Class distribution before SMOTE
print("Training class distribution before SMOTE:")
print(y_train.value_counts())

## 9. Apply SMOTE

SMOTE (Synthetic Minority Oversampling Technique) creates **synthetic examples** of the minority class (fraud) by interpolating between existing fraud examples. This helps the model learn the patterns of fraudulent transactions better.

We apply SMOTE **only to the training data**. The test data must stay untouched so that our evaluation reflects real-world conditions.

Because IEEE-CIS is a large dataset, we use `sampling_strategy=0.2` instead of forcing a full 50/50 balance. This creates enough synthetic fraud examples to help the model, without making the dataset unrealistically balanced.

In [ ]:
# Apply SMOTE only to the training data
smote = SMOTE(
    sampling_strategy=0.2,
    random_state=1
)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)

In [ ]:
print("Training class distribution before SMOTE:")
print(y_train.value_counts())

print("Training class distribution after SMOTE:")
print(y_train_smote.value_counts())

## 10. Create XGBoost Model

We now train an `XGBClassifier` for binary classification. The settings below are reasonable defaults for this kind of imbalanced classification problem, without being overly complex.

In [ ]:
# Create XGBoost model
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    random_state=1,
    n_jobs=-1
)

In [ ]:
# Train the model on the SMOTE-resampled training data
xgb_model.fit(
    X_train_smote,
    y_train_smote
)

## 11. Make Predictions

Instead of only using `predict()`, we first get the predicted **fraud probabilities**. This lets us try different decision thresholds later, instead of being locked into the default 0.5 threshold.

In [ ]:
# Predicted probability of fraud (class 1) for each test transaction
y_prob = xgb_model.predict_proba(X_test)[:, 1]

# Predictions using the standard 0.5 threshold
y_pred = (y_prob >= 0.5).astype(int)

## 12. Evaluate the Default 0.5 Threshold

We evaluate the model using several metrics, not just accuracy, since accuracy can be misleading on an imbalanced dataset.

In [ ]:
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("Precision:")
print(precision_score(y_test, y_pred, zero_division=0))

print("Recall:")
print(recall_score(y_test, y_pred, zero_division=0))

print("F1 Score:")
print(f1_score(y_test, y_pred, zero_division=0))

print("ROC-AUC:")
print(roc_auc_score(y_test, y_prob))

print("PR-AUC:")
print(average_precision_score(y_test, y_prob))

print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

## 13. Tune the Decision Threshold

The default threshold of 0.5 is not always the best choice for imbalanced problems like fraud detection. Here we test several thresholds and compare precision, recall, and F1 score for each one.

In [ ]:
# Thresholds to test
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

results = []

for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    precision_t = precision_score(y_test, y_pred_t, zero_division=0)
    recall_t = recall_score(y_test, y_pred_t, zero_division=0)
    f1_t = f1_score(y_test, y_pred_t, zero_division=0)
    results.append([t, precision_t, recall_t, f1_t])

threshold_results = pd.DataFrame(
    results,
    columns=["Threshold", "Precision", "Recall", "F1 Score"]
)

print(threshold_results)

## 14. Plot Threshold Results

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(threshold_results["Threshold"], threshold_results["Precision"], marker="o", label="Precision")
plt.plot(threshold_results["Threshold"], threshold_results["Recall"], marker="o", label="Recall")
plt.plot(threshold_results["Threshold"], threshold_results["F1 Score"], marker="o", label="F1 Score")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision, Recall and F1 Score vs Decision Threshold")
plt.legend()
plt.grid(True)
plt.show()

## 15. Select the Best Threshold

We select the threshold that gives the highest F1 score, and use it to create the final predictions.

In [ ]:
best_threshold = threshold_results.loc[
    threshold_results["F1 Score"].idxmax(),
    "Threshold"
]

print("Best Decision Threshold:", best_threshold)

In [ ]:
# Final predictions using the best threshold
final_predictions = (
    y_prob >= best_threshold
).astype(int)

## 16. Evaluate the Tuned Threshold

In [ ]:
print("Confusion Matrix (Tuned Threshold):")
print(confusion_matrix(y_test, final_predictions))

precision_tuned = precision_score(y_test, final_predictions, zero_division=0)
recall_tuned = recall_score(y_test, final_predictions, zero_division=0)
f1_tuned = f1_score(y_test, final_predictions, zero_division=0)

print("Precision:", precision_tuned)
print("Recall:", recall_tuned)
print("F1 Score:", f1_tuned)

print("Classification Report (Tuned Threshold):")
print(classification_report(y_test, final_predictions, zero_division=0))

## Save Predictions to CSV

We save the final (tuned-threshold) predictions to `predictions.csv`, along with the matching `TransactionID`, the actual label, and the predicted fraud probability. This makes the output easier to review than just the raw predicted class.

In [ ]:
# Save final predictions to a CSV file
predictions_df = pd.DataFrame({
    "TransactionID": fraud_data.loc[X_test.index, "TransactionID"],
    "Actual": y_test,
    "Predicted": final_predictions,
    "Fraud_Probability": y_prob
})

predictions_df.to_csv("predictions.csv", index=False)
print("Predictions saved to predictions.csv")

In [ ]:
# Compare default 0.5 threshold with the tuned threshold
precision_default = precision_score(y_test, y_pred, zero_division=0)
recall_default = recall_score(y_test, y_pred, zero_division=0)
f1_default = f1_score(y_test, y_pred, zero_division=0)

comparison_table = pd.DataFrame({
    "Metric": ["Precision", "Recall", "F1 Score"],
    "Threshold 0.5": [precision_default, recall_default, f1_default],
    "Tuned Threshold": [precision_tuned, recall_tuned, f1_tuned]
})

print(comparison_table)

## 17. Feature Importance

We use XGBoost's built-in feature importance to see which features the model relied on the most when making predictions.

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": xgb_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance.head(20))

## 18. Plot Feature Importance

In [ ]:
top_features = feature_importance.head(20)

plt.figure(figsize=(10, 8))
plt.barh(top_features["Feature"], top_features["Importance"])
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.title("Top 20 Feature Importances (XGBoost)")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

## 19. Interpretation

**Why is the dataset imbalanced?**
Fraudulent transactions are naturally rare compared to legitimate ones, so the `isFraud` classes are very unequal in size.

**Why is accuracy alone not enough?**
Because fraud is so rare, a model could predict "not fraud" for every transaction and still get very high accuracy, while completely failing to catch any fraud. That is why we also look at precision, recall, F1, ROC-AUC and PR-AUC.

**What does SMOTE do?**
SMOTE creates synthetic examples of the minority (fraud) class by interpolating between existing fraud examples in the training data. This gives the model more fraud examples to learn from, without touching the test data.

**Why is XGBoost suitable here?**
XGBoost is a tree-based ensemble model that handles complex, non-linear relationships and mixed types of features well, and it performs strongly on structured/tabular data like this transaction dataset.

**Why does the decision threshold matter?**
The default threshold of 0.5 is not always optimal, especially for imbalanced problems. Lowering or raising the threshold changes the trade-off between catching more fraud (recall) and avoiding false alarms (precision).

**What does precision mean here?**
Precision is the percentage of transactions flagged as fraud that were actually fraud. High precision means fewer legitimate transactions are wrongly flagged.

**What does recall mean here?**
Recall is the percentage of actual fraud cases that the model successfully caught. High recall means fewer fraud cases are missed.

**Why is PR-AUC useful for imbalanced data?**
PR-AUC focuses on the performance on the minority (positive/fraud) class, and is less affected by the large number of true negatives than ROC-AUC, which makes it more informative for heavily imbalanced problems like this one.

**What do the feature importance results indicate?**
The top features are the ones the XGBoost model relied on most heavily to separate fraud from non-fraud transactions. This does **not** mean these features *cause* fraud — it only means the model found them useful for making predictions.

## 20. Final Conclusion

In this notebook, we applied XGBoost to the IEEE-CIS credit card fraud detection dataset. Because the dataset was heavily imbalanced, we used SMOTE on the training data only to give the model more fraud examples to learn from. Instead of relying on the default 0.5 decision threshold, we tested several thresholds and selected the one that gave the best F1 score. We evaluated the model using accuracy, precision, recall, F1 score, ROC-AUC and PR-AUC rather than accuracy alone, and finally used XGBoost's feature importance scores to understand which features the model relied on most when identifying fraudulent transactions.